# M0 — Intégration d'un modèle sur étagère (DiagOps)

## Mission

Ce notebook documente les **choix techniques**, la **stack**, le **contrat de données** et l'**évaluation** du module 0.

Objectif M0 : exposer une API qui transforme une **note technicien** en diagnostic JSON structuré, **sans fine-tuning** — le modèle est appelé via Hugging Face Inference.

> Ce notebook est une **synthèse** (choix + démonstration). Le code applicatif est dans `app/`, `ui/` et `tests/`.

## 1. Stack technique et langages

| Couche | Technologie | Pourquoi |
|--------|-------------|----------|
| Langage | **Python 3.11+** | Écosystème ML/API, aligné avec le cursus |
| API REST | **FastAPI** | Validation auto, OpenAPI `/docs`, async-ready |
| Contrats | **Pydantic v2** | Schémas entrée/sortie typés (`DiagnoseRequest`, `DiagnoseResponse`) |
| Modèle | **Qwen2.5-7B-Instruct** (HF Inference) | JSON structuré en français sans GPU local |
| Client HF | **huggingface_hub.InferenceClient** | Appel HTTP managé, timeout 60 s |
| UI exploratoire | **Streamlit** | Tester rapidement les rapports du data pack |
| UI légère | **HTML/JS** (`ui/index.html`) | Démo sans framework lourd |
| Tests | **pytest** + mocks HF | CI sans token réseau |
| Config | **python-dotenv** (`.env`) | Secrets hors git (`HF_TOKEN`) |

### Architecture

```
technicien → POST /diagnose → FastAPI → model_client → HF Inference
                                    ↓
                            Pydantic (validation JSON)
                                    ↓
                            DiagnoseResponse
```

**Pas de base de données** en M0 : une requête = un appel stateless.

## 2. Données — pas de nettoyage pipeline

M0 **ne nettoie pas** de tables industrielles. La seule source est :

```
data_pack/2026-S1/reports/reports.jsonl
```

Chaque ligne = un rapport avec `report_id`, `equipment_id`, `technician_note`, etc.

| Étape | Traitement |
|-------|------------|
| Lecture | JSONL ligne par ligne |
| Entrée API | `technician_note` (min 1 caractère, strip) |
| Sortie | Normalisation sévérité FR→EN, confiance bornée [0,1] |
| Garde-fou | `requires_human_review=true` si confiance < 0.6 ou note < 40 car. |

Le **nettoyage massif** des données équipements/événements commence en **M2**.

## 0. Environnement

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd

pd.set_option('display.max_colwidth', 80)

def find_repo_root() -> Path:
    for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (root / 'data_pack' / 'MANIFEST.yaml').is_file():
            return root
    raise RuntimeError('Racine dépôt introuvable — lancez depuis work/M0 ou définissez DIAGOPS_DATA_DIR')

REPO = find_repo_root()
M0_DIR = REPO / 'work' / 'M0'
DATA_DIR = Path(os.environ.get('DIAGOPS_DATA_DIR', REPO / 'data_pack' / '2026-S1'))
REPORTS_PATH = DATA_DIR / 'reports' / 'reports.jsonl'

print('Dépôt :', REPO)
print('M0    :', M0_DIR)
print('Data  :', REPORTS_PATH, '→', REPORTS_PATH.exists())

## 3. Explorer les rapports source

In [ ]:
def load_reports(path: Path, limit: int | None = None) -> pd.DataFrame:
    rows = []
    with path.open(encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            if limit and i >= limit:
                break
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

if REPORTS_PATH.exists():
    df_reports = load_reports(REPORTS_PATH)
    print(f'{len(df_reports)} rapports chargés')
    display(df_reports[['report_id', 'equipment_id']].head())
    print('\nLongueur moyenne des notes :', df_reports['technician_note'].str.len().mean().round(0), 'caractères')
else:
    print('Fichier reports.jsonl absent — vérifiez le data_pack')

## 4. Contrat API (schémas Pydantic)

Fichiers : `app/schemas.py`, `app/main.py`, `app/model_client.py`

In [ ]:
import sys
sys.path.insert(0, str(M0_DIR))

from app.schemas import DiagnoseRequest, DiagnoseResponse

example_in = DiagnoseRequest(
    report_id='RPT-2026S1-0001',
    technician_note='Pompe P-204 : vibration au démarrage, bruit métallique.',
    equipment_id='EQ-PUMP-001',
)
print('Entrée validée :', example_in.model_dump())
print('\nChamps sortie obligatoires :', list(DiagnoseResponse.model_fields.keys()))

## 5. Choix du modèle et providers

| Provider (`MODEL_PROVIDER`) | Usage |
|----------------------------|-------|
| `hf_api` (défaut M0) | API Hugging Face — Mac sans GPU |
| `local_baseline` | Qwen3-0.6B local (pont vers M1) |
| `local_lora` | Qwen3-0.6B + adaptateur LoRA M1 |

**Prompt système** : contraint la sortie JSON, interdit le markdown, impose le français.

**Post-traitement** : mapping sévérité (`élevé` → `high`), clamp confiance, revue humaine forcée.

## 6. Gestion des erreurs HTTP

| Code | Cause |
|------|-------|
| 422 | Note vide, JSON invalide côté client |
| 502 | HF indisponible ou JSON modèle non parseable |
| 503 | `HF_TOKEN` manquant |

Les tests dans `tests/test_api.py` mockent HF pour couvrir ces cas sans réseau.

## 7. Évaluation (aperçu)

Résultats détaillés : [`evaluation_m0.md`](../evaluation_m0.md)

Synthèse des limites observées (2026-08-03) :

In [ ]:
eval_summary = pd.DataFrame([
    {'constat': 'JSON conforme au contrat', 'statut': 'OK'},
    {'constat': 'Sévérité souvent lissée à medium', 'statut': 'Limite'},
    {'constat': 'Cas ambigu → confiance basse + revue humaine', 'statut': 'OK'},
    {'constat': 'Pas de données capteur/historique', 'statut': 'Hors périmètre M0'},
    {'constat': 'Latence HF ~1–2 s/appel', 'statut': 'Acceptable dev'},
])
eval_summary

## 8. Appel API live (optionnel)

Prérequis : API démarrée (`uvicorn app.main:app --reload --app-dir .`) et `HF_TOKEN` dans `.env`.

In [ ]:
import urllib.request
import urllib.error

API_URL = os.environ.get('DIAGOPS_API_URL', 'http://127.0.0.1:8000')

def call_diagnose(report_id: str, note: str, equipment_id: str | None = None) -> dict:
    payload = json.dumps({
        'report_id': report_id,
        'technician_note': note,
        'equipment_id': equipment_id,
    }).encode()
    req = urllib.request.Request(
        f'{API_URL}/diagnose',
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=90) as resp:
        return json.loads(resp.read())

try:
    health = urllib.request.urlopen(f'{API_URL}/health', timeout=3)
    print('API OK :', health.read().decode())
    if REPORTS_PATH.exists():
        sample = load_reports(REPORTS_PATH, limit=1).iloc[0]
        result = call_diagnose(sample['report_id'], sample['technician_note'], sample.get('equipment_id'))
        print(json.dumps(result, indent=2, ensure_ascii=False))
except urllib.error.URLError as e:
    print('API non joignable — démarrez uvicorn (voir README.md)')
    print(e)

## 9. Commandes utiles

```bash
cd work/M0 && source .venv/bin/activate
PYTHONPATH=. pytest tests/ -v
uvicorn app.main:app --reload --app-dir .
streamlit run ui/streamlit_app.py
```

## 10. Passage vers M1

- M0 prouve l'intégration **zero-shot**.
- M1 spécialise **Qwen3-0.6B** en LoRA sur `diagops_train.jsonl`.
- Le provider `local_lora` permet de réutiliser l'API M0 avec l'adaptateur entraîné.